# 10 — Supplementary Analysis: Per-Capita Waste & Hunger-Waste Connection

Integrates Census population and USDA food insecurity data with ReFED food waste data to produce per-capita analysis and hunger-waste connection visualizations.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')

SOFT_COLORS = {
    'primary': '#4A6B6F',      # slate teal
    'secondary': '#8FB3A8',    # sage green
    'accent': '#C4836D',       # warm terracotta
    'highlight': '#D4C97A',    # muted gold
    'neutral': '#A0A0A0',      # medium gray
    'bg': '#F8F6F3',           # warm off-white
    'text': '#2D3436',         # near-black
}
SECTOR_COLORS = {
    'Farm': '#7BA17D',
    'Foodservice': '#D4836D',
    'Manufacturing': '#5B8BA0',
    'Residential': '#D4C97A',
    'Retail': '#6BADA0',
}

os.makedirs('outputs/analysis/charts', exist_ok=True)
print('Setup complete.')

Setup complete.


## Section 1: Load & Merge Data

In [3]:
# --- Load all 4 datasets ---
pop = pd.read_csv('data/census_state_population_2024.csv')
insec = pd.read_csv('data/usda_food_insecurity_2022_2024.csv')
state_waste = pd.read_parquet('outputs/analysis/cleaned_data/state_summary.parquet')
policy = pd.read_csv('data/policy_timeline.csv')

print('Population:', pop.shape)
print('Food insecurity:', insec.shape)
print('State waste:', state_waste.shape)
print('Policy timeline:', policy.shape)

Population: (50, 2)
Food insecurity: (50, 3)
State waste: (216930, 32)
Policy timeline: (12, 4)


In [4]:
# --- Filter to 2024 and aggregate across sectors/food types ---
waste_2024 = state_waste[state_waste['year'] == 2024].groupby('state').agg(
    tons_waste=('tons_waste', 'sum'),
    tons_landfill=('tons_landfill', 'sum'),
    us_dollars_surplus=('us_dollars_surplus', 'sum'),
    meals_wasted=('meals_wasted', 'sum'),
    tons_surplus=('tons_surplus', 'sum'),
    tons_donations=('tons_donations', 'sum'),
).reset_index()

print('Aggregated 2024 state waste:', waste_2024.shape)
waste_2024.head(3)

Aggregated 2024 state waste: (50, 7)


,state,tons_waste,tons_landfill,us_dollars_surplus,meals_wasted,tons_surplus,tons_donations
0,Alabama,6.527950e+05,400187.867858,5.448215e+09,1.220057e+09,7.510602e+05,19025.954063
1,Alaska,7.866108e+04,53320.728856,8.309915e+08,1.499654e+08,9.272249e+04,2743.249038
2,Arizona,1.849348e+06,707055.912878,9.026390e+09,3.208284e+09,1.957735e+06,32764.211898


In [5]:
# --- Merge all datasets ---
df = waste_2024.merge(pop, on='state', how='inner')
df = df.merge(insec, on='state', how='inner')

# Ban status from policy_timeline
ban_states = set(policy['state'].dropna().unique())
df['has_ban'] = df['state'].isin(ban_states)

# --- Compute per-capita metrics ---
df['tons_waste_per_capita'] = df['tons_waste'] / df['population_2024']
df['tons_landfill_per_capita'] = df['tons_landfill'] / df['population_2024']
df['dollars_surplus_per_capita'] = df['us_dollars_surplus'] / df['population_2024']
df['meals_wasted_per_capita'] = df['meals_wasted'] / df['population_2024']

print(f'Merged dataset: {df.shape[0]} states')
print(f'Ban states in merged data: {df["has_ban"].sum()}')
df[['state', 'tons_waste_per_capita', 'meals_wasted_per_capita', 'has_ban']].head(3)

Merged dataset: 50 states
Ban states in merged data: 12


,state,tons_waste_per_capita,meals_wasted_per_capita,has_ban
0,Alabama,0.126567,236.550652,False
1,Alaska,0.106280,202.619540,False
2,Arizona,0.243901,423.123438,False


## Section 2: Per-Capita Waste by State (Chart 1)

In [6]:
plot_df = df.sort_values('tons_waste_per_capita', ascending=True).reset_index(drop=True)
bar_colors = [SOFT_COLORS['primary'] if b else SOFT_COLORS['neutral'] for b in plot_df['has_ban']]

fig, ax = plt.subplots(figsize=(10, 16), facecolor='white')
ax.set_facecolor(SOFT_COLORS['bg'])

ax.barh(plot_df['state'], plot_df['tons_waste_per_capita'], color=bar_colors, edgecolor='white', linewidth=0.3)

ax.set_xlabel('Tons of Food Waste Per Capita (2024)', fontsize=12, color=SOFT_COLORS['text'])
ax.set_title('Per-Capita Food Waste by State (2024)', fontsize=16, fontweight='bold',
             color=SOFT_COLORS['text'], pad=15)
ax.tick_params(axis='y', labelsize=8, colors=SOFT_COLORS['text'])
ax.tick_params(axis='x', labelsize=10, colors=SOFT_COLORS['text'])
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=SOFT_COLORS['primary'], label='Organic Waste Ban'),
    Patch(facecolor=SOFT_COLORS['neutral'], label='No Ban'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10, framealpha=0.9)

for spine in ax.spines.values():
    spine.set_visible(False)

plt.savefig('outputs/analysis/charts/per_capita_waste_by_state.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved to outputs/analysis/charts/per_capita_waste_by_state.png')

Saved to outputs/analysis/charts/per_capita_waste_by_state.png


## Section 3: Food Insecurity vs Food Waste Scatter (Chart 2)

In [7]:
fig, ax = plt.subplots(figsize=(12, 8), facecolor='white')
ax.set_facecolor(SOFT_COLORS['bg'])

# Bubble sizes scaled by population
size_scale = df['population_2024'] / df['population_2024'].max() * 600

# Plot ban vs non-ban separately
for has_ban, label, color in [(True, 'Organic Waste Ban', SOFT_COLORS['primary']),
                               (False, 'No Ban', SOFT_COLORS['neutral'])]:
    mask = df['has_ban'] == has_ban
    ax.scatter(df.loc[mask, 'food_insecurity_pct'],
               df.loc[mask, 'tons_waste_per_capita'],
               s=size_scale[mask], alpha=0.6, color=color, edgecolors='white',
               linewidth=0.5, label=label, zorder=3)

# Trend line (linear regression)
slope, intercept, r_value, p_value, std_err = stats.linregress(
    df['food_insecurity_pct'], df['tons_waste_per_capita'])
x_line = np.linspace(df['food_insecurity_pct'].min(), df['food_insecurity_pct'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=SOFT_COLORS['accent'],
        linewidth=2, linestyle='--', alpha=0.8, label=f'Trend (R²={r_value**2:.3f})', zorder=2)

# Annotate notable states
annotate_states = ['Texas', 'California', 'Arkansas', 'Mississippi', 'North Dakota']
for _, row in df[df['state'].isin(annotate_states)].iterrows():
    ax.annotate(row['state'], (row['food_insecurity_pct'], row['tons_waste_per_capita']),
                fontsize=9, fontweight='bold', color=SOFT_COLORS['text'],
                textcoords='offset points', xytext=(8, 5),
                arrowprops=dict(arrowstyle='->', color=SOFT_COLORS['text'], lw=0.8))

ax.set_xlabel('Food Insecurity Rate (%)', fontsize=12, color=SOFT_COLORS['text'])
ax.set_ylabel('Tons of Food Waste Per Capita', fontsize=12, color=SOFT_COLORS['text'])
ax.set_title('Food Insecurity Rate vs Per-Capita Food Waste (2024)',
             fontsize=15, fontweight='bold', color=SOFT_COLORS['text'], pad=15)
ax.legend(fontsize=10, framealpha=0.9, loc='upper left')
ax.grid(alpha=0.2)
for spine in ax.spines.values():
    spine.set_visible(False)

plt.savefig('outputs/analysis/charts/insecurity_vs_waste.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved to outputs/analysis/charts/insecurity_vs_waste.png')

Saved to outputs/analysis/charts/insecurity_vs_waste.png


## Section 4: Meals Wasted vs Food Insecure Population (Chart 3)

In [8]:
# Compute food insecure population
df['food_insecure_people'] = df['population_2024'] * df['food_insecurity_pct'] / 100

# Top 10 states by food insecure population
top10 = df.nlargest(10, 'food_insecure_people').sort_values('food_insecure_people', ascending=True).copy()
top10['meals_per_insecure_person'] = top10['meals_wasted'] / top10['food_insecure_people']

fig, ax = plt.subplots(figsize=(12, 8), facecolor='white')
ax.set_facecolor(SOFT_COLORS['bg'])

y_pos = np.arange(len(top10))
bar_height = 0.35

# Scale meals_wasted to millions and food_insecure_people to millions for readability
meals_m = top10['meals_wasted'] / 1e6
insecure_m = top10['food_insecure_people'] / 1e6

bars1 = ax.barh(y_pos + bar_height/2, meals_m, bar_height,
                color=SOFT_COLORS['accent'], label='Meals Wasted (millions)', edgecolor='white', linewidth=0.3)
bars2 = ax.barh(y_pos - bar_height/2, insecure_m, bar_height,
                color=SOFT_COLORS['primary'], label='Food Insecure People (millions)', edgecolor='white', linewidth=0.3)

# Annotate ratio on each bar
for i, (_, row) in enumerate(top10.iterrows()):
    ratio = row['meals_per_insecure_person']
    ax.annotate(f'{ratio:,.0f} meals/person',
                xy=(meals_m.iloc[i], y_pos[i] + bar_height/2),
                xytext=(5, 0), textcoords='offset points',
                fontsize=8, color=SOFT_COLORS['text'], va='center', fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(top10['state'], fontsize=10, color=SOFT_COLORS['text'])
ax.set_xlabel('Count (millions)', fontsize=12, color=SOFT_COLORS['text'])
ax.set_title('Meals Wasted vs Food Insecure Population\nTop 10 States by Food Insecure Population (2024)',
             fontsize=14, fontweight='bold', color=SOFT_COLORS['text'], pad=15)
ax.legend(fontsize=10, loc='lower right', framealpha=0.9)
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)

plt.savefig('outputs/analysis/charts/meals_vs_insecurity.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved to outputs/analysis/charts/meals_vs_insecurity.png')

Saved to outputs/analysis/charts/meals_vs_insecurity.png


## Section 5: Summary Statistics

In [9]:
# --- Key findings ---
total_meals_wasted = df['meals_wasted'].sum()
total_food_insecure = df['food_insecure_people'].sum()
meals_per_insecure = total_meals_wasted / total_food_insecure

highest_pc_state = df.loc[df['tons_waste_per_capita'].idxmax()]
lowest_pc_state = df.loc[df['tons_waste_per_capita'].idxmin()]

corr, corr_p = stats.pearsonr(df['food_insecurity_pct'], df['tons_waste_per_capita'])

ban_mean = df.loc[df['has_ban'], 'tons_waste_per_capita'].mean()
no_ban_mean = df.loc[~df['has_ban'], 'tons_waste_per_capita'].mean()

print(f'=== National Summary ===')
print(f'Total meals wasted: {total_meals_wasted/1e9:.1f}B')
print(f'Total food insecure people: {total_food_insecure/1e6:.1f}M')
print(f'Meals wasted per food-insecure person: {meals_per_insecure:,.0f}')
print()
print(f'Highest per-capita waste: {highest_pc_state["state"]} ({highest_pc_state["tons_waste_per_capita"]:.4f} tons/person)')
print(f'Lowest per-capita waste: {lowest_pc_state["state"]} ({lowest_pc_state["tons_waste_per_capita"]:.4f} tons/person)')
print()
print(f'Correlation (food insecurity % vs per-capita waste): r={corr:.3f}, p={corr_p:.4f}')
print()
print(f'Mean per-capita waste — Ban states: {ban_mean:.4f} tons | Non-ban states: {no_ban_mean:.4f} tons')
print(f'Difference: {((no_ban_mean - ban_mean) / no_ban_mean) * 100:.1f}% lower in ban states')

=== National Summary ===
Total meals wasted: 114.9B
Total food insecure people: 45.4M
Meals wasted per food-insecure person: 2,531

Highest per-capita waste: Wisconsin (0.4487 tons/person)
Lowest per-capita waste: Rhode Island (0.1018 tons/person)

Correlation (food insecurity % vs per-capita waste): r=-0.172, p=0.2315

Mean per-capita waste — Ban states: 0.1738 tons | Non-ban states: 0.1646 tons
Difference: -5.6% lower in ban states


In [10]:
# --- Save summary to JSON ---
summary = {
    'national': {
        'total_meals_wasted': round(total_meals_wasted),
        'total_food_insecure_people': round(total_food_insecure),
        'meals_per_food_insecure_person': round(meals_per_insecure),
    },
    'highest_per_capita_waste': {
        'state': highest_pc_state['state'],
        'tons_per_capita': round(float(highest_pc_state['tons_waste_per_capita']), 4),
    },
    'lowest_per_capita_waste': {
        'state': lowest_pc_state['state'],
        'tons_per_capita': round(float(lowest_pc_state['tons_waste_per_capita']), 4),
    },
    'insecurity_waste_correlation': {
        'pearson_r': round(corr, 3),
        'p_value': round(corr_p, 4),
    },
    'ban_vs_no_ban': {
        'ban_states_mean_tons_per_capita': round(ban_mean, 4),
        'no_ban_states_mean_tons_per_capita': round(no_ban_mean, 4),
        'pct_lower_in_ban_states': round(((no_ban_mean - ban_mean) / no_ban_mean) * 100, 1),
    },
    'charts_produced': [
        'outputs/analysis/charts/per_capita_waste_by_state.png',
        'outputs/analysis/charts/insecurity_vs_waste.png',
        'outputs/analysis/charts/meals_vs_insecurity.png',
    ]
}

with open('outputs/analysis/supplementary_findings.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved to outputs/analysis/supplementary_findings.json')

Saved to outputs/analysis/supplementary_findings.json
